In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata

In [ ]:
import joblib

# Load the object back
emb = joblib.load("./data/larry/larry_flowmap_embedder.pkl")

print("Loaded object:", type(emb))

In [ ]:
from flowmap.geometry.fixed_points import FixedPointAnalyzer

# ---------------------------------------------------
# Run fixed point analysis
# ---------------------------------------------------
fpa = FixedPointAnalyzer(emb)

fp_info = fpa.identify_fixed_points(
    grid_resolution=60,
    speed_smoothing=1.7,
    speed_quantile_threshold=0.05,
    jacobian_radius=0.1,
)

print(f"Found {len(fp_info)} fixed points")

for i, info in enumerate(fp_info):
    print(f"\nFixed point {i}")
    print("  position:", info["position"])
    print("  type:", info["type"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.neighbors import NearestNeighbors
from flowmap.utils import compute_velocity_on_grid

fp1 = fp_info[0]["position"]   # Fixed point 1 (source)
fp2 = fp_info[2]["position"]   # Fixed point 2 (saddle)

# ------------------------------------------------------------
# Helper: remove grid seeds outside the manifold
# ------------------------------------------------------------
def points_inside_mask(X_emb, seeds, k=8, radius_scale=1.2):
    nn = NearestNeighbors(n_neighbors=k).fit(X_emb)
    r = np.median(nn.kneighbors(X_emb)[0][:, -1]) * radius_scale
    neigh_idx = nn.radius_neighbors(seeds, radius=r, return_distance=False)
    return np.array([len(ix) > 0 for ix in neigh_idx])


# ------------------------------------------------------------
# Helper: manually add arrows
# ------------------------------------------------------------
def add_manual_arrows(ax, coords, X_emb, V_pred):
    nn = NearestNeighbors(n_neighbors=1).fit(X_emb)
    _, idx = nn.kneighbors(coords)
    idx = idx.ravel()

    ax.quiver(
        X_emb[idx,0], X_emb[idx,1],
        V_pred[idx,0], V_pred[idx,1],
        angles="xy",
        scale_units="xy",
        scale=3,
        width=0.003,
        headwidth=4.5,
        headlength=4.0,
        headaxislength=2.3,
        minlength=0.2,
        color="k",
        alpha=0.9,
    )


# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
X_emb = emb.X_emb
labels = np.asarray(adata.obs["state_info"].values)


# ------------------------------------------------------------
# Colors
# ------------------------------------------------------------
uniq = np.unique(labels)
other = [u for u in uniq if u != "Undifferentiated"]

cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#d3d3d3"

cell_colors = np.array([colmap[l] for l in labels])


# ------------------------------------------------------------
# Velocity grid
# ------------------------------------------------------------
Xg, keep_mass, _ = compute_velocity_on_grid(X_emb, grid_size=25, min_mass=0.01)

keep_inside = points_inside_mask(X_emb, Xg)
Xg = Xg[keep_inside]

Vg = emb.spline_vf.predict(Xg)

def remove_grid_arrows(Xg, Vg, coords):
    """
    Remove arrows from the velocity grid by snapping to nearest grid seed.
    """
    if len(coords) == 0:
        return Xg, Vg

    nn = NearestNeighbors(n_neighbors=1).fit(Xg)
    _, idx = nn.kneighbors(coords)
    idx = np.unique(idx.ravel())

    keep = np.ones(len(Xg), dtype=bool)
    keep[idx] = False

    return Xg[keep], Vg[keep]

remove_coords = [
    [12.6,12.5]
]

Xg, Vg = remove_grid_arrows(Xg, Vg, remove_coords)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12,12))

# background cells
undiff = labels == "Undifferentiated"
ax.scatter(
    X_emb[undiff,0], X_emb[undiff,1],
    c="#d3d3d3",
    s=40,
    alpha=0.2,
    linewidths=0,
)

# foreground cells
mask = ~undiff
ax.scatter(
    X_emb[mask,0], X_emb[mask,1],
    c=cell_colors[mask],
    s=80,
    alpha=0.4,
    linewidths=0,
)

# velocity arrows
ax.quiver(
    Xg[:,0], Xg[:,1],
    Vg[:,0], Vg[:,1],
    angles="xy",
    scale_units="xy",
    scale=3,
    width=0.003,
    headwidth=4.5,
    headlength=4.0,
    headaxislength=2.3,
    minlength=0.2,
    color="k",
    alpha=0.9,
)

# --------------------------------------------------------------------------
# Fixed points (explicit, no indices)
# --------------------------------------------------------------------------
dx, dy = -0.02, -0.03

# Fixed point 1
ax.scatter(fp1[0], fp1[1], color="red", s=1000, zorder=4)
ax.text(
    fp1[0] + dx, fp1[1] + dy, "1",
    color="white", fontsize=36, weight="bold",
    ha="center", va="center", zorder=5
)

# Fixed point 2
ax.scatter(fp2[0], fp2[1], color="red", s=1000, zorder=4)
ax.text(
    fp2[0] + dx, fp2[1] + dy, "2",
    color="white", fontsize=36, weight="bold",
    ha="center", va="center", zorder=5
)

# ------------------------------------------------------------
# Manual arrow patches
# ------------------------------------------------------------
patch_coords = [
    [12.0,10.4],
    [14,3.6],
    [12.5,3.6],
    [12.6,12.5]
]

V_cells = emb.spline_vf.predict(X_emb)

add_manual_arrows(ax, patch_coords, X_emb, V_cells)


# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig(
    "./figures/larry/larry_embedding.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from flowmap import *
import flowmap

coords = np.array([
    [-6.0,6.0],
    [2.5,-5.6],
    fp2
])

# ---------------------------------------------------
# Initialize optimizer
# ---------------------------------------------------
lap = flowmap.geometry.LagrangianPathOptimizer(
    embedding=emb,
    D=1.0,
    lam=1e-4,
)

root = fp1

results = []
for i, end in enumerate(coords, start=1):
    print(f"\n=== Path {i}: {root} → {end} ===")

    res = lap.fit_path(
        start=root,
        end=end,
        distance_mode="orig",   # or "embed"
        subsample_n=4000,
        k=20,
        alpha=0.0,
        n_segments=100,
        lr=5e-3,
        iters=300,
    )

    results.append(res)

# Collect refined paths
refined_paths = [r["path_refined"] for r in results]

In [ ]:
def plot_fixed_point_panel_with_paths(
    fp_idx,
    fp,
    X_emb,
    spline_vf,
    adata,
    refined_paths,
    epsilon=0.20,
    nx=20,
    ny=20,
    stream_density=0.7,
    arrowsize=1.5,
    line_scale=2.5,
    savepath=None,
    title=None,
    path_color="red",
    path_alpha=0.8,
    path_style="--",
):
    emb_range = np.ptp(X_emb, axis=0)
    x0, y0 = map(float, fp)

    eps_x = epsilon * emb_range[0]
    eps_y = epsilon * emb_range[1]
    xlim = (x0 - eps_x, x0 + eps_x)
    ylim = (y0 - eps_y, y0 + eps_y)

    # --- Subset cells ---
    mask = (
        (X_emb[:, 0] >= xlim[0]) & (X_emb[:, 0] <= xlim[1]) &
        (X_emb[:, 1] >= ylim[0]) & (X_emb[:, 1] <= ylim[1])
    )
    X_sub = X_emb[mask]

    if X_sub.size == 0:
        print(f"[⚠️] No points near FP{fp_idx} ({x0:.2f}, {y0:.2f}). Skipping.")
        return None

    # --- Cell coloring ---
    labels = np.asarray(adata.obs["state_info"].values)
    uniq = np.unique(labels)
    other = [lab for lab in uniq if lab != "Undifferentiated"]

    cmap = plt.get_cmap("tab10", len(other))
    colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
    colmap["Undifferentiated"] = "#d3d3d3"

    labels_sub = labels[mask]
    colors_sub = np.array([colmap[lab] for lab in labels_sub])

    undiff_mask = labels_sub == "Undifferentiated"
    diff_mask = ~undiff_mask

    # --- Velocity grid ---
    Xg, keep, Vg, meshes = compute_velocity_on_grid(
        X_sub,
        spline_vf=spline_vf,
        grid_size=nx,
        grid_density=1.0,
        min_mass=0.01,
        return_mesh=True,
    )

    xx, yy = meshes
    ny_, nx_ = yy.shape[0], xx.shape[1]

    gx = np.linspace(xx[0, 0], xx[0, -1], nx_, dtype=float)
    gy = np.linspace(yy[0, 0], yy[-1, 0], ny_, dtype=float)

    Vx = np.full((ny_, nx_), np.nan)
    Vy = np.full((ny_, nx_), np.nan)

    dx = (gx[-1] - gx[0]) / (nx_ - 1)
    dy = (gy[-1] - gy[0]) / (ny_ - 1)

    j_idx = np.clip(np.rint((Xg[:, 0] - gx[0]) / dx).astype(int), 0, nx_ - 1)
    i_idx = np.clip(np.rint((Xg[:, 1] - gy[0]) / dy).astype(int), 0, ny_ - 1)

    Vx[i_idx, j_idx] = Vg[:, 0]
    Vy[i_idx, j_idx] = Vg[:, 1]

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8, 7), facecolor="white")

    # Background (undiff)
    if np.any(undiff_mask):
        ax.scatter(
            X_sub[undiff_mask, 0],
            X_sub[undiff_mask, 1],
            color="#d3d3d3",
            s=70,
            alpha=0.5,
            edgecolors="none",
            zorder=1,
        )

    # Foreground (diff)
    if np.any(diff_mask):
        ax.scatter(
            X_sub[diff_mask, 0],
            X_sub[diff_mask, 1],
            color=colors_sub[diff_mask],
            s=70,
            alpha=0.8,
            edgecolors="none",
            zorder=2,
        )

    # --- LAP paths ---
    for path in refined_paths:
        ax.plot(
            path[:, 0],
            path[:, 1],
            path_style,
            lw=9,
            color=path_color,
            alpha=path_alpha,
            zorder=2.5,
        )

    # --- Vector field (quiver) ---
    skip = (slice(None, None, 2), slice(None, None, 2))
    ax.quiver(
        xx[skip],
        yy[skip],
        np.nan_to_num(Vx[skip]),
        np.nan_to_num(Vy[skip]),
        angles="xy",
        scale_units="xy",
        scale=line_scale,
        width=0.006,
        headwidth=5.0,
        headlength=4.5,
        color="k",
        alpha=0.9,
        zorder=3,
    )

    # --- Fixed point marker ---
    dx_text = -0.03
    dy_text = -0.08

    ax.scatter(x0, y0, color="red", s=2000, zorder=4)
    ax.text(
        x0 + dx_text,
        y0 + dy_text,
        str(fp_idx),
        color="white",
        fontsize=50,
        weight="bold",
        ha="center",
        va="center",
        zorder=5,
    )

    # --- Aesthetics ---
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

    ax.set_title(title or f"Fixed Point {fp_idx}", fontsize=36, pad=12)

    plt.tight_layout()

    if savepath:
        plt.savefig(savepath, dpi=300, bbox_inches="tight")
        print(f"[Saved] {savepath}")

    plt.show()

    return fig, ax


plot_fixed_point_panel_with_paths(
    fp_idx=1,
    fp=fp1,
    X_emb=emb.X_emb,
    spline_vf=emb.spline_vf,
    adata=adata,
    refined_paths=refined_paths,
    epsilon=0.3,
    path_alpha=0.9,
    path_color="darkorange",
    path_style="-",
    title="Fixed Point 1: Source",
    savepath="./figures/larry/fp1_source.pdf",
)

In [ ]:
import time

t0 = time.time()
emb.fit_gene_level_splines(dof_gene=80, dof_vf_gene=80)
print(f"[TIME] Gene spline fit: {time.time() - t0:.2f}s")

t0 = time.time()
analyzer = flowmap.geometry.GeneGradientAnalyzer(emb)
print(f"[TIME] Analyzer init (Jacobians): {time.time() - t0:.2f}s")

In [ ]:
import numpy as np
import time

from flowmap.core.spline import Spline


# --------------------------------------------------
# 1. Generate embedding-like data
# --------------------------------------------------
N = 30000     # like emb.X_emb size
d = 2
D = 10       # output dim (e.g. PCs / genes)

np.random.seed(0)

X_emb = np.random.randn(N, d)
Y = np.random.randn(N, D)


# --------------------------------------------------
# 2. Fit spline (same as pipeline)
# --------------------------------------------------
spline = Spline(X_emb, n_control_points=1000)
spline.fit(Y, dof=50)


# --------------------------------------------------
# 3. Attach fast version
# --------------------------------------------------
def compute_jacobians_fast(self, evaluation_points, eps=1e-12):
    Xev = np.asarray(evaluation_points, float)

    centers = self.centers
    alpha = self.radial_coefficients
    beta = self.polynomial_coefficients

    M, d = Xev.shape
    m = centers.shape[0]
    D = alpha.shape[1]

    # (M, m, d)
    diffs = Xev[:, None, :] - centers[None, :, :]

    s = np.sum(diffs * diffs, axis=2)

    factor = np.zeros_like(s)
    mask = s > eps
    if np.any(mask):
        factor[mask] = s[mask] * (2 * np.log(s[mask]) + 1)

    # (M, m, d)
    weighted = factor[:, :, None] * diffs

    # reshape → (m, M*d)
    weighted_2d = weighted.transpose(1, 0, 2).reshape(m, M * d)

    # GEMM
    out = alpha.T @ weighted_2d

    J_rbf = out.reshape(D, M, d).transpose(1, 0, 2)

    # polynomial
    dPdx, _ = self._poly_derivatives(Xev)
    J_poly = np.einsum("mpd,pk->mkd", dPdx, beta)

    return J_rbf + J_poly


# monkey patch
spline.compute_jacobians_fast = compute_jacobians_fast.__get__(spline)


# --------------------------------------------------
# 4. Sample evaluation points (like emb.X_emb subset)
# --------------------------------------------------
idx = np.random.choice(N, size=2000, replace=False)
X_test = X_emb[idx]


# --------------------------------------------------
# 5. Benchmark
# --------------------------------------------------
def time_fn(fn, X, n_runs=3):
    times = []
    for _ in range(n_runs):
        t0 = time.time()
        out = fn(X)
        t1 = time.time()
        times.append(t1 - t0)
    return np.mean(times), out


t_old, J_old = time_fn(spline.compute_jacobians, X_test)
t_fast, J_fast = time_fn(spline.compute_jacobians_fast, X_test)


# --------------------------------------------------
# 6. Compare results
# --------------------------------------------------
max_diff = np.max(np.abs(J_old - J_fast))
rel_diff = max_diff / (np.max(np.abs(J_old)) + 1e-12)

print("=== Timing ===")
print(f"old (einsum) : {t_old:.4f} s")
print(f"fast (GEMM)  : {t_fast:.4f} s")
print(f"speedup      : {t_old / t_fast:.2f}x")

print("\n=== Accuracy ===")
print(f"max abs diff : {max_diff:.3e}")
print(f"rel diff     : {rel_diff:.3e}")

In [ ]:
import numpy as np
import time

from flowmap.core.spline import Spline


# --------------------------------------------------
# 1. Generate embedding-like data
# --------------------------------------------------
N = 10000
d = 2
D = 10

np.random.seed(0)

X_emb = np.random.randn(N, d)
Y = np.random.randn(N, D)


# --------------------------------------------------
# 2. Fit spline
# --------------------------------------------------
spline = Spline(X_emb, n_control_points=1000)
spline.fit(Y, dof=50)


# --------------------------------------------------
# 3. Optimized Hessian
# --------------------------------------------------
def compute_hessians_fast(self, evaluation_points, eps=1e-12):
    Xev = np.asarray(evaluation_points, float)

    centers = self.centers
    alpha = self.radial_coefficients
    beta = self.polynomial_coefficients

    M, d = Xev.shape
    D = alpha.shape[1]

    # (M, m, d)
    diffs = Xev[:, None, :] - centers[None, :, :]
    s = np.sum(diffs * diffs, axis=2)

    factor_dd = np.zeros_like(s)
    factor_I = np.zeros_like(s)

    mask = s > eps
    if np.any(mask):
        log_s = np.log(s[mask])
        factor_dd[mask] = 2.0 * (2.0 * log_s + 3.0)
        factor_I[mask] = s[mask] * (2.0 * log_s + 1.0)

    # --------------------------------------------------
    # TERM 2 (fast GEMM)
    # --------------------------------------------------
    term2 = factor_I @ alpha   # (M, D)

    # --------------------------------------------------
    # TERM 1 (d^2 GEMMs, no mnij tensor)
    # --------------------------------------------------
    term1 = np.zeros((M, D, d, d))

    for i in range(d):
        for j in range(d):
            contrib = factor_dd * diffs[:, :, i] * diffs[:, :, j]  # (M, m)
            term1[:, :, i, j] = contrib @ alpha                   # GEMM

    I = np.eye(d)
    H_rbf = term1 + term2[:, :, None, None] * I

    # polynomial (constant)
    _, d2P = self._poly_derivatives(Xev)
    H_poly = np.einsum("pij,pk->kij", d2P, beta)
    H_poly = np.broadcast_to(H_poly[None], (M, D, d, d))

    return H_rbf + H_poly


# attach
spline.compute_hessians_fast = compute_hessians_fast.__get__(spline)


# --------------------------------------------------
# 4. Sample evaluation points (like emb.X_emb)
# --------------------------------------------------
idx = np.random.choice(N, size=2000, replace=False)
X_test = X_emb[idx]


# --------------------------------------------------
# 5. Benchmark helper
# --------------------------------------------------
def time_fn(fn, X, n_runs=3):
    times = []
    for _ in range(n_runs):
        t0 = time.time()
        out = fn(X)
        t1 = time.time()
        times.append(t1 - t0)
    return np.mean(times), out


# --------------------------------------------------
# 6. Run benchmark
# --------------------------------------------------
t_new, H_new = time_fn(spline.compute_hessians, X_test)
t_fast, H_fast = time_fn(spline.compute_hessians_fast, X_test)


# --------------------------------------------------
# 7. Compare
# --------------------------------------------------
max_diff = np.max(np.abs(H_new - H_fast))
rel_diff = max_diff / (np.max(np.abs(H_new)) + 1e-12)

print("=== Timing ===")
print(f"einsum version : {t_new:.4f} s")
print(f"fast version   : {t_fast:.4f} s")
print(f"speedup        : {t_new / t_fast:.2f}x")

print("\n=== Accuracy ===")
print(f"max abs diff : {max_diff:.3e}")
print(f"rel diff     : {rel_diff:.3e}")

In [ ]:
# emb.fit_gene_level_splines(dof_gene=80, dof_vf_gene=80)

# analyzer = flowmap.geometry.GeneGradientAnalyzer(emb)

# results_all = []

# # Precompute once (important!)
# n_genes = analyzer.J.shape[1]
# gene_indices = np.arange(n_genes)
# gene_names = getattr(emb, "gene_names", None)

# for trajectory in refined_paths:

#     # --------------------------------------------------
#     # 1) Find neighboring cells along trajectory
#     # --------------------------------------------------
#     neighbor_indices = analyzer.find_neighbors(
#         trajectory,
#         epsilon=0.04,
#     )

#     if len(neighbor_indices) == 0:
#         continue

#     # --------------------------------------------------
#     # 2) Compute gene gradient alignment
#     # --------------------------------------------------
#     grad_res = analyzer.compute_relative_gradients(
#         neighbor_indices,
#         gene_indices,
#         weight="magnitude",   # good default
#     )

#     # --------------------------------------------------
#     # 3) Store results
#     # --------------------------------------------------
#     results_all.append({
#         "trajectory": trajectory,
#         "neighbor_indices": neighbor_indices,
#         "gene_indices": gene_indices,
#         "gene_names": gene_names,
#         "angles": grad_res["angles"],
#         "magnitudes": grad_res["magnitudes"],
#     })